# YouTube Trending Videos — Silver Layer Transformation
### Bronze → Cleansed Star Schema (Dimensions + Facts)

**Purpose:** This notebook reads the raw bronze video and category tables and transforms them into a clean, typed, analytics-ready **star schema**. Invalid rows are filtered, duplicates removed, and derived metrics (engagement rate, like ratio, days-to-trend) are computed.

**Transformations Applied:**
- Filter invalid rows (`_bronze_is_valid = false`)
- Parse `trending_date` from `yy.dd.MM` format to proper `DateType`
- Deduplicate on `(video_id, trending_date)` keeping the highest-view row
- Derive engagement metrics: `engagement_rate`, `like_ratio`, `days_to_trend`
- Clean channel/title strings (strip special characters)

**Tables Produced:**

| Type | Table | Grain | Description |
|------|-------|-------|-------------|
| Dimension | `dim_date` | One row per calendar date | Calendar attributes from all trending dates |
| Dimension | `dim_category` | One row per YouTube category | Category ID → name lookup (exploded from JSON) |
| Dimension | `dim_channel` | One row per channel | Lifetime channel aggregates (views, likes, trending count) |
| Dimension | `dim_video` | One row per video | Video metadata (title, tags, thumbnail, description) |
| Dimension | `dim_tags` | One row per unique tag | Exploded and cleaned tag lookup |
| Bridge | `bridge_video_tags` | One row per (video, tag) pair | Many-to-many video ↔ tag relationship |
| Fact | `fact_trending_videos` | One row per video per trending day | Core metrics: views, likes, engagement, partitioned by category |
| Fact | `fact_video_trending_trajectory` | One row per video per trending day | Day-over-day deltas: view velocity, like acceleration |
| Fact | `fact_channel_daily_performance` | One row per channel per trending day | Daily channel rollup: total views, avg engagement |

---
_Upstream: [bronze](#) · Downstream: [gold](#)_

In [0]:
# ── PySpark imports ──
# functions.* → column-level operations (date parsing, regex, aggregations)
# Window      → deduplication and trajectory ranking
# types       → explicit casting for IDs, booleans, timestamps

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, StringType, BooleanType, TimestampType

In [0]:
# ── Medallion architecture paths (ADLS Gen2) ──
# YouTube data shares the same storage account as employee data,
# but lives under separate UC schemas (bronze_youtube, silver_youtube, gold_youtube).

root_path = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net"
bronze_path = f"{root_path}/bronze"
silver_path = f"{root_path}/silver"
gold_path = f"{root_path}/gold"

# ── Unity Catalog schema references ──
bronze_sch = "bronze_youtube"
silver_sch = "silver_youtube"
gold_sch = "gold_youtube"
youtube_db = "employeedatacatalog"

In [0]:
# ── Load both bronze sources ──
# videos   → ~40K rows of US YouTube trending video stats (CSV origin)
# category → nested JSON with category ID → name mappings

df_video = spark.read.format("delta").load("abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net/bronze/videos")
df_category = spark.read.format("delta").load("abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net/bronze/category")


In [0]:
# ============================================================
# Video data cleansing & feature engineering
# ============================================================
# Key transformations:
#   1. Filter out invalid rows (bronze quality flag)
#   2. Parse trending_date from "yy.dd.MM" format to proper DateType
#   3. Standardize comments_disabled to boolean
#   4. Derive three engagement metrics:
#      - engagement_rate: (likes + dislikes + comments) / views * 100
#      - like_ratio: likes / (likes + dislikes) * 100
#      - days_to_trend: days between publish and trending
#   5. Clean channel_title and title (strip special characters)
#   6. Drop bronze audit columns no longer needed

df_video_silver = df_video.filter(col("_bronze_is_valid") == 'true')\
    .withColumn("trending_date", to_date(col("trending_date"), "yy.dd.MM"))\
    .withColumn("comments_disabled", when(upper(col('comments_disabled')) == 'TRUE',True).otherwise(False))\
    .withColumn("engagement_rate",                                # (likes+dislikes+comments) / views * 100
        round((col("likes")+col("dislikes")+col("comment_count"))/nullif(col("views"),lit(0)) *100,3))\
    .withColumn("like_ratio",                                     # likes / (likes+dislikes) * 100
        round(col("likes")/nullif(col("likes")+col("dislikes"),lit(0)) *100,3))\
    .withColumn("days_to_trend",                                  # How fast did it go trending?
        date_diff(col("trending_date"),to_date(col("publish_time"))))\
    .withColumn("channel_title",regexp_replace(col("channel_title"),"[^a-zA-Z0-9]",''))\
    .withColumn("title",regexp_replace(col("title"),"[^a-zA-Z0-9]",''))\
    .drop("bronze_ingested_at","_bronze_source_file","_bronze_batch_id"
        "_bronze_is_valid","publish_time")


In [0]:
# ============================================================
# Remove duplicates within the same (video_id, trending_date)
# ============================================================
# A video can appear multiple times on the same trending date
# (e.g., different snapshot times). We keep the row with the
# highest view count and discard the rest.

WINDOW_FUNCT = Window.partitionBy("video_id", "trending_date").orderBy(col("VIEWS").desc())

df_depduplicate = df_video_silver \
    .withColumn("rw", row_number().over(WINDOW_FUNCT)) \
    .filter(col("rw") == 1) \
    .drop("rw") \
    .withColumn("_silver_ingested_at", current_timestamp())       # Audit timestamp for silver layer


In [0]:
# ============================================================
# DIMENSION: dim_date
# ============================================================
# Builds a calendar dimension from all distinct trending dates.
# Enriched with day/month/quarter attributes, weekend flags,
# and a placeholder is_holiday column for future enhancement.

# Step 1 — Extract all unique trending dates
df_dates = df_depduplicate.filter(col("trending_date").isNotNull()) \
    .select(col("trending_date").alias("dt")).distinct()

# Step 2 — Enrich with calendar attributes
df_dim_date = df_dates \
    .withColumn("date_key", date_format(col("dt"), "yyyyMMdd").cast(IntegerType())) \
    .withColumn("date", col("dt")) \
    .withColumn("day", dayofmonth(col("dt"))) \
    .withColumn("month", month(col("dt"))) \
    .withColumn("year", year(col("dt"))) \
    .withColumn("week", weekofyear(col("dt"))) \
    .withColumn("day_of_week", dayofweek(col("dt"))) \
    .withColumn("day_name", date_format(col("dt"), "E")) \
    .withColumn("month_name", date_format(col("dt"), "MMMM")) \
    .withColumn("quarter", quarter(col("dt"))) \
    .withColumn("is_weekend",                                     # Sat=7, Sun=1
        when((col("day_of_week") == 1) | (col("day_of_week") == 7), True).otherwise(False)) \
    .withColumn("is_holiday", lit(False))                         # Placeholder for future holiday calendar \
    .withColumn("is_weekday",
        when(~((col("day_of_week") == 1) | (col("day_of_week") == 7)), True).otherwise(False)) \
    .drop("dt") \
    .orderBy(col("date_key"))

# Persist to silver Delta and register in UC
df_write = df_dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_path}/dim_date")

df_metadata = spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {youtube_db}.{silver_sch}.dim_date_new
    USING DELTA
    LOCATION '{silver_path}/dim_date'
""")

In [0]:
# ============================================================
# DIMENSION: dim_category
# ============================================================
# The raw category JSON has a nested "items" array. We explode it
# into flat rows, extract the snippet fields, and cast to proper types.
# Result: one row per category (Entertainment, Music, News, etc.)

# Step 1 — Explode the nested JSON array into individual rows
df_dim_category_flat = df_category.select(
    explode(col("items")).alias("item"),
    col("_bronze_ingested_at"),
    col("_bronze_source_file")
).select(
    col("item.id").alias("category_id"),
    col("item.snippet.title").alias("category_name"),
    col("item.snippet.assignable").alias("assignable"),
    col("item.snippet.channelId").alias("source_channel_id"),
    col("_bronze_ingested_at"),
    col("_bronze_source_file")
)

# Step 2 — Cast types and swap audit column to silver timestamp
df_dim_category = df_dim_category_flat \
    .withColumn("category_key", col("category_id").cast(IntegerType())) \
    .withColumn("category_name", col("category_name").cast(StringType())) \
    .withColumn("assignable", col("assignable").cast(BooleanType())) \
    .withColumn("_silver_ingested_at", col("_bronze_ingested_at").cast(TimestampType())) \
    .drop("category_id", "source_channel_id", "_bronze_ingested_at", "_bronze_source_file")

# Persist to silver Delta and register in UC
df_write_cat = df_dim_category.write.format("delta").mode("overwrite").save(f"{silver_path}/dim_category")

df_metadata_cat = spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {youtube_db}.{silver_sch}.dim_category
    USING DELTA
    LOCATION '{silver_path}/dim_category'
""")

display(df_dim_category)

In [0]:
# ============================================================
# DIMENSION: dim_channel
# ============================================================
# Aggregates video-level data to one row per channel.
# Captures lifetime totals (views, likes, trending count) and
# date range of trending activity. Useful for channel profiling.

# Step 1 — Group by channel and compute lifetime metrics
df_dim_channel = df_depduplicate.groupBy("channel_title").agg(
    count("video_id").alias("total_trending_videos"),              # How many times their videos trended
    sum("views").alias("total_views"),
    sum("likes").alias("total_likes"),
    sum("dislikes").alias("total_dislikes"),
    sum("comment_count").alias("total_comments"),
    min("trending_date").alias("first_trending_date"),             # When they first appeared
    max("trending_date").alias("last_trending_date"),              # Most recent trending date
    countDistinct("channel_title").alias("num_categories_used")    # Content diversity
)

# Step 2 — Add surrogate key and derived columns
df_dim_channel = df_dim_channel \
    .withColumn("channel_key", md5(col("channel_title")))         # Deterministic surrogate key \
    .withColumn("avg_views_per_video",                            # Average views per trending appearance
        round(col("total_views") / col("total_trending_videos"), 0)) \
    .withColumn("_silver_ingested_at", current_timestamp()) \
    .withColumn("_record_version", lit(1))                        # For future SCD tracking

# Persist to silver Delta and register in UC
df_df_dim_channel = df_dim_channel.write.format("delta").mode("overwrite").save(f"{silver_path}/dim_channel")

df_metadata_dim_channel = spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {youtube_db}.{silver_sch}.dim_channel
    USING DELTA
    LOCATION '{silver_path}/dim_channel'
""")

In [0]:
# ============================================================
# DIMENSION: dim_video
# ============================================================
# One row per video with metadata attributes.
# Links to dim_channel via channel_key.
# Description is truncated to 500 chars to keep the table lean.

df_dim_video = df_depduplicate \
    .withColumn("video_key", md5("video_id"))                    # Deterministic surrogate key \
    .withColumn("channel_key", md5(col("channel_title")))        # FK to dim_channel \
    .withColumn("_silver_ingested_at", current_timestamp()) \
    .select(
        "video_key", "video_id", "channel_key",
        col("title").alias("video_title"),
        "category_id", "thumbnail_link", "tags",
        "comments_disabled", "ratings_disabled", "video_error_or_removed",
        substring(col("description"), 1, 500).alias("description_snippet")  # Truncate for storage
    )

# Persist to silver Delta and register in UC
df_dim_video_channel = df_dim_video.write.format("delta").mode("overwrite").save(f"{silver_path}/dim_video")

df_metadata_dim_video = spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {youtube_db}.{silver_sch}.dim_video
    USING DELTA
    LOCATION '{silver_path}/dim_video'
""")


In [0]:
# ============================================================
# FACT: fact_trending_videos
# ============================================================
# The primary fact table — one row per video per trending day.
# Joins to dim_date (via trending_date_key) and carries all
# engagement metrics. Partitioned by category_key for query perf.

# Step 1 — Prepare date key lookups for join
trending_date_key = df_dim_date.select(
    col("date_key").alias("trending_date_key"),
    col("date").alias("_tdate")
)

publish_date_key = df_dim_date.select(
    col("date_key").alias("publish_date_key"),
    col("date").alias("_pdate")
)

# Step 2 — Build the fact table with surrogate keys and date FKs
df_fact = df_depduplicate \
    .withColumn("video_key", md5("video_id")) \
    .withColumn("channel_key", md5(col("channel_title"))) \
    .withColumn("fact_key",                                       # Unique row identifier
        md5(concat_ws("_", col("video_id"), col("trending_date")))) \
    .join(trending_date_key,                                      # Join to get date FK
          df_depduplicate["trending_date"] == trending_date_key["_tdate"]) \
    .select(
        "fact_key", "video_key", "channel_key",
        col("category_id").cast(IntegerType()).alias("category_key"),
        "trending_date_key", "video_id", "trending_date",
        "views", "likes", "dislikes", "comment_count",
        "engagement_rate", "like_ratio", "days_to_trend",
        "comments_disabled", "ratings_disabled", "video_error_or_removed",
        "_silver_ingested_at"
    )

# Persist to silver Delta, partitioned by category for fast filtered queries
df_fact = df_fact.write.format("delta").mode("overwrite") \
    .partitionBy("category_key") \
    .save(f"{silver_path}/fact_trending_videos")

df_metadata_df_fact = spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {youtube_db}.{silver_sch}.fact_trending_videos
    USING DELTA
    LOCATION '{silver_path}/fact_trending_videos'
""")


In [0]:
# ============================================================
# DIMENSION: dim_tags
# ============================================================
# YouTube tags are stored as a single pipe-delimited string per video.
# We explode them into individual rows, clean whitespace and quotes,
# then deduplicate to build a reusable tag lookup.
#
# This enables: tag popularity analysis, trending tag detection,
# tag co-occurrence networks, category-tag correlation.

# Step 1 — Explode pipe-separated tags and clean up
df_tags_exploded = (
    df_depduplicate
    .filter(col("tags").isNotNull() & (col("tags") != "[none]"))
    .select(
        md5("video_id").alias("video_key"),
        explode(split(col("tags"), "\\|")).alias("raw_tag")        # Split on pipe delimiter
    )
    .withColumn("tag_name",                                       # Clean: lowercase, strip quotes & whitespace
        lower(trim(regexp_replace(col("raw_tag"), '"', ''))))
    .filter(col("tag_name") != "")                                # Drop empties after cleanup
)

# Step 2 — Build the unique tag dimension with a surrogate key
dim_tags = (
    df_tags_exploded
    .select("tag_name")
    .distinct()
    .withColumn("tag_key", md5(col("tag_name")))                  # Deterministic surrogate key
    .withColumn("tag_length", length(col("tag_name")))            # Useful for filtering junk tags
    .withColumn("_silver_ingested_at", current_timestamp())
    .select("tag_key", "tag_name", "tag_length", "_silver_ingested_at")
)

print(f"Unique tags: {dim_tags.count():,}")

# Persist to silver Delta and register in UC
dim_tags.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_path}/dim_tags")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{silver_sch}.dim_tags
          USING DELTA LOCATION '{silver_path}/dim_tags'""")

dim_tags.limit(10).display()

In [0]:
# ============================================================
# BRIDGE: bridge_video_tags
# ============================================================
# Links videos to their tags (many-to-many). A single video can
# have 20+ tags, and a popular tag appears across thousands of videos.
#
# Grain: one row per (video_key, tag_key) pair.
# Use case: "Which tags appear most often on trending videos?"
#           "What tags co-occur with 'music'?"

bridge_video_tags = df_tags_exploded \
    .withColumn("tag_key", md5(lower(trim(regexp_replace(col("raw_tag"), '"', ''))))) \
    .select("video_key", "tag_key") \
    .distinct() \
    .withColumn("_silver_ingested_at", current_timestamp())

print(f"Video-tag pairs: {bridge_video_tags.count():,}")

# Persist to silver Delta and register in UC
bridge_video_tags.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_path}/bridge_video_tags")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{silver_sch}.bridge_video_tags
          USING DELTA LOCATION '{silver_path}/bridge_video_tags'""")

bridge_video_tags.limit(10).display()

In [0]:
# ============================================================
# FACT: fact_video_trending_trajectory
# ============================================================
# For videos that trend multiple days, this table tracks how their
# metrics evolve over time. Using window functions, we compute:
#   - daily deltas (how many new views/likes since yesterday)
#   - trending day number (1st day, 2nd day, etc.)
#   - cumulative trending streak length
#
# 5,644 videos trend for >1 day — this captures their growth curve.

# Step 1 — Window: partition by video, order by trending date
w_trajectory = Window.partitionBy("video_id").orderBy("trending_date")

# Step 2 — Compute lagged values and deltas
df_trajectory = df_depduplicate \
    .withColumn("video_key", md5("video_id")) \
    .withColumn("trending_day_num",                               # Which day of trending is this?
        row_number().over(w_trajectory)) \
    .withColumn("prev_views", lag("views", 1).over(w_trajectory)) \
    .withColumn("prev_likes", lag("likes", 1).over(w_trajectory)) \
    .withColumn("prev_dislikes", lag("dislikes", 1).over(w_trajectory)) \
    .withColumn("prev_comments", lag("comment_count", 1).over(w_trajectory)) \
    .withColumn("views_delta",                                    # New views since last trending day
        col("views") - coalesce(col("prev_views"), lit(0))) \
    .withColumn("likes_delta",
        col("likes") - coalesce(col("prev_likes"), lit(0))) \
    .withColumn("dislikes_delta",
        col("dislikes") - coalesce(col("prev_dislikes"), lit(0))) \
    .withColumn("comments_delta",
        col("comment_count") - coalesce(col("prev_comments"), lit(0))) \
    .withColumn("views_growth_pct",                               # % growth from prior day
        round(col("views_delta") / nullif(col("prev_views"), lit(0)) * 100, 2)) \
    .withColumn("trajectory_key",                                 # Surrogate key
        md5(concat_ws("_", col("video_id"), col("trending_date").cast("string")))) \
    .select(
        "trajectory_key", "video_key", "video_id", "trending_date",
        "trending_day_num", "views", "likes", "dislikes", "comment_count",
        "views_delta", "likes_delta", "dislikes_delta", "comments_delta",
        "views_growth_pct", "engagement_rate", "like_ratio",
        "_silver_ingested_at"
    )

print(f"Trajectory rows: {df_trajectory.count():,}")

# Persist to silver Delta and register in UC
df_trajectory.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_path}/fact_video_trending_trajectory")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{silver_sch}.fact_video_trending_trajectory
          USING DELTA LOCATION '{silver_path}/fact_video_trending_trajectory'""")

df_trajectory.filter(col("trending_day_num") > 1).orderBy("video_id", "trending_date").limit(10).display()

In [0]:
# ============================================================
# FACT: fact_channel_daily_performance
# ============================================================
# Aggregates video-level trending data up to channel + date grain.
# Answers: "How did this channel perform on a given trending day?"
#
# Useful for: channel growth curves, daily leaderboards,
#             identifying channels that spike vs steady performers.

df_channel_daily = df_depduplicate \
    .withColumn("channel_key", md5(col("channel_title"))) \
    .groupBy("channel_key", "channel_title", "trending_date") \
    .agg(
        count("video_id").alias("trending_video_count"),            # How many of their videos trended today?
        sum("views").alias("daily_total_views"),
        sum("likes").alias("daily_total_likes"),
        sum("dislikes").alias("daily_total_dislikes"),
        sum("comment_count").alias("daily_total_comments"),
        round(avg("engagement_rate"), 3).alias("avg_engagement_rate"),
        round(avg("like_ratio"), 3).alias("avg_like_ratio"),
        max("views").alias("top_video_views"),                     # Their best-performing video today
        countDistinct("category_id").alias("categories_represented")  # How diverse is their content?
    ) \
    .withColumn("daily_performance_key",                           # Surrogate key
        md5(concat_ws("_", col("channel_key"), col("trending_date").cast("string")))) \
    .withColumn("_silver_ingested_at", current_timestamp()) \
    .select(
        "daily_performance_key", "channel_key", "channel_title", "trending_date",
        "trending_video_count", "daily_total_views", "daily_total_likes",
        "daily_total_dislikes", "daily_total_comments",
        "avg_engagement_rate", "avg_like_ratio",
        "top_video_views", "categories_represented",
        "_silver_ingested_at"
    )

print(f"Channel-day rows: {df_channel_daily.count():,}")

# Persist to silver Delta and register in UC
df_channel_daily.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{silver_path}/fact_channel_daily_performance")

spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{silver_sch}.fact_channel_daily_performance
          USING DELTA LOCATION '{silver_path}/fact_channel_daily_performance'""")

df_channel_daily.orderBy(desc("daily_total_views")).limit(10).display()